# Chapter 14: Physics-Based Rendering

Source orientation: printed pages 357-382; physical PDF pages 374-399. This notebook uses that span only to choose the concepts and terminology. The explanations, code, diagrams, and synthetic images below are original course material.

## Chapter Goal

Physics-based rendering turns light transport into quantities that can be measured, conserved, sampled, and debugged. By the end of this notebook you should be able to connect photon energy, radiance, Fresnel reflection, refraction, Beer attenuation, BRDF normalization, the rendering equation, and Monte Carlo path tracing to concrete arrays and checks.

The chapter's main warning is that plausible pictures are not enough. A renderer can look reasonable while losing energy, using the wrong projected-area factor, confusing radiance with irradiance, or sampling a BRDF with a mismatched probability density. The notebook therefore pairs every visual with a numerical or symbolic invariant.


## Translation guide

| Chapter idea | Computational translation | Invariant to keep visible |
| --- | --- | --- |
| Photon packet | A wavelength-indexed energy sample with `q = h c / lambda`. | Shorter visible wavelengths carry more energy per photon. |
| Spectral energy and power | Histograms or density samples over wavelength and time. | Doubling bin width doubles bin energy but should not change the underlying density estimate much. |
| Irradiance | Radiance integrated over incident directions with a projected-area factor. | Constant incident radiance gives `H = pi L`. |
| Radiance | Power density per projected area and solid angle. | In empty space radiance is invariant along a ray, even though flux per area changes. |
| Fresnel and refraction | Interface rules: Schlick reflectance, Snell direction, and total internal reflection. | Reflectance stays in `[R0, 1]`; transmitted directions satisfy Snell's law when they exist. |
| Beer attenuation | Per-channel exponential transmittance through a distance. | `I(s) = I(0) a**s` is monotone when channel transmittance is below one. |
| BRDF | A function that maps incident direction and outgoing direction to reflected radiance per irradiance. | Directional hemispherical reflectance must not exceed one. |
| Rendering equation | A recursive integral over directions, or over visible surface area with geometry and visibility factors. | Visibility and the two cosine terms can zero out a contribution. |
| Path tracing | A Monte Carlo estimator for the rendering equation. | The estimator divides by the sampling PDF; better matching PDFs reduce variance. |


## Visual storyboard

| Visual artifact | Concept | Library route | What to inspect | Check |
| --- | --- | --- | --- | --- |
| `photon-energy-sensor-bands.png` | Photons, spectral energy, RGB sensor filters | NumPy and Matplotlib for unit-aware 2D plots | Energy per photon falls with wavelength; bandpass filters turn spectra into RGB counters. | Energy monotonicity and positive band totals. |
| `radiance-irradiance-geometry.png` | Radiance invariance and irradiance integration | SymPy for `pi` identity, Matplotlib for geometry | Area growth and inverse-square falloff cancel for radiance; cosine projection creates `H = pi L`. | Symbolic and numeric hemisphere integrals. |
| `fresnel-refraction-beer.png` | Fresnel, Snell refraction, total internal reflection, Beer law | NumPy and Matplotlib | Grazing angles approach full reflection; glass-to-air can lose the transmitted ray; colored glass filters by distance. | Schlick bounds, Snell residual, TIR threshold, monotone attenuation. |
| `brdf-energy-lobes.png` and `brdf-lobe-explorer.html` | BRDF lobes and energy normalization | Matplotlib plus Plotly HTML for inspectable lobe geometry | A narrow glossy lobe can still conserve the same total reflected energy. | Numeric directional hemispherical reflectance. |
| `rendering-equation-visibility-geometry.png` | Directional and area forms of the rendering equation | Matplotlib and NetworkX | Surface contribution depends on BRDF, incoming radiance, visibility, distance, and both cosines. | Geometry factor positive, occluded, and back-facing cases. |
| `monte-carlo-hemisphere-convergence.png` | Monte Carlo estimator variance | NumPy, Pandas, Matplotlib | Cosine-weighted directions match the integrand better than uniform hemisphere samples. | Final estimates and RMSE comparison. |
| `path-traced-diffuse-synthetic-room.png` | Recursive path tracing on a synthetic scene | NumPy ray geometry and Matplotlib display | More samples reduce path tracing noise while preserving the same transport model. | Finite nonblank image, ray count, luminance statistics. |

All paths are book-local under `artifacts/chapter-14/`. The static figures are durable PNGs; the BRDF lobe explorer is standalone Plotly HTML; numeric summaries are JSON or CSV checks.


In [ ]:
from __future__ import annotations

from pathlib import Path
import math
import sys

CHAPTER = 14
TITLE = "Physics-Based Rendering"
TOPIC = f"chapter-{CHAPTER:02d}"
PRINTED_PAGES = "357-382"
PDF_PAGES = "374-399"

search_roots = [Path.cwd(), *Path.cwd().parents]
for root in list(search_roots):
    search_roots.extend([p for p in [root / "Fundamentals-of-Computer-Graphics"] if p.exists()])

BOOK_ROOT = None
for candidate in search_roots:
    if (candidate / "00-book-index.ipynb").exists() and (candidate / "utils").exists():
        BOOK_ROOT = candidate.resolve()
        break
if BOOK_ROOT is None:
    raise RuntimeError("Could not find the FCG book root")

if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

ARTIFACT_ROOT = BOOK_ROOT / "artifacts" / TOPIC
for kind in ["figures", "html", "checks", "tables", "data"]:
    (ARTIFACT_ROOT / kind).mkdir(parents=True, exist_ok=True)

print(f"Book root: {BOOK_ROOT.name}")
print(f"Artifact root: {ARTIFACT_ROOT.relative_to(BOOK_ROOT).as_posix()}")


In [ ]:
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import sympy as sp

from utils.artifacts import (
    assert_artifacts,
    book_relative,
    display_artifact,
    save_json,
    save_matplotlib,
    save_plotly_html,
    save_table_csv,
)
from utils.notebook_checks import assert_nonblank_image
from utils.plotting import PALETTE, close, style_axis

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "#fbfcfe",
    "font.size": 9,
})

figure_paths: list[Path] = []
html_paths: list[Path] = []
check_paths: list[Path] = []
table_paths: list[Path] = []


def remember(path: Path, bucket: list[Path]) -> Path:
    bucket.append(path)
    return path


def normalize(v: np.ndarray) -> np.ndarray:
    arr = np.asarray(v, dtype=float)
    n = float(np.linalg.norm(arr))
    if n == 0.0:
        raise ValueError("zero vector cannot be normalized")
    return arr / n


def photon_energy_joule(wavelength_nm: np.ndarray | float) -> np.ndarray:
    h = 6.62607015e-34
    c = 299_792_458.0
    return h * c / (np.asarray(wavelength_nm, dtype=float) * 1e-9)


def r0_from_indices(ni: float, nt: float) -> float:
    return float(((nt - ni) / (nt + ni)) ** 2)


def schlick(cos_theta: np.ndarray | float, r0: np.ndarray | float) -> np.ndarray:
    c = np.clip(np.asarray(cos_theta, dtype=float), 0.0, 1.0)
    return np.asarray(r0, dtype=float) + (1.0 - np.asarray(r0, dtype=float)) * (1.0 - c) ** 5


def refract_direction(direction: np.ndarray, normal_to_incident_medium: np.ndarray, ni: float, nt: float) -> tuple[np.ndarray | None, float]:
    d = normalize(direction)
    n = normalize(normal_to_incident_medium)
    cos_theta_i = -float(np.dot(d, n))
    eta = ni / nt
    k = 1.0 - eta * eta * (1.0 - cos_theta_i * cos_theta_i)
    if k < 0.0:
        return None, k
    transmitted = eta * d + (eta * cos_theta_i - math.sqrt(k)) * n
    return normalize(transmitted), k


def beer_transmission(unit_transmittance: np.ndarray, distance: np.ndarray | float) -> np.ndarray:
    return np.asarray(unit_transmittance, dtype=float) ** np.asarray(distance, dtype=float)[..., None]


def normalized_cosine_lobe(theta: np.ndarray, reflectance: float, exponent: float) -> np.ndarray:
    return reflectance * (exponent + 2.0) / (2.0 * np.pi) * np.clip(np.cos(theta), 0.0, 1.0) ** exponent


def directional_hemispherical_reflectance(theta: np.ndarray, rho: np.ndarray) -> float:
    return float(2.0 * np.pi * np.trapezoid(rho * np.cos(theta) * np.sin(theta), theta))


def make_onb(normal: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    n = normalize(normal)
    helper = np.array([0.0, 0.0, 1.0]) if abs(n[2]) < 0.95 else np.array([0.0, 1.0, 0.0])
    tangent = normalize(np.cross(helper, n))
    bitangent = np.cross(n, tangent)
    return tangent, bitangent, n


def cosine_sample_hemisphere(rng: np.random.Generator, normal: np.ndarray) -> np.ndarray:
    u1, u2 = rng.random(2)
    r = math.sqrt(u1)
    phi = 2.0 * math.pi * u2
    local = np.array([r * math.cos(phi), r * math.sin(phi), math.sqrt(max(0.0, 1.0 - u1))])
    tangent, bitangent, n = make_onb(normal)
    return normalize(local[0] * tangent + local[1] * bitangent + local[2] * n)


In [ ]:
storyboard = {
    "chapter": CHAPTER,
    "source_span": {"printed_pages": PRINTED_PAGES, "pdf_pages": PDF_PAGES},
    "library_routing": {
        "numpy": "unit calculations, vector optics, Monte Carlo samples, and the small path tracer",
        "sympy": "exact hemisphere and Lambertian normalization identities",
        "matplotlib": "durable annotated diagrams and convergence plots",
        "plotly": "standalone 3D BRDF lobe explorer",
        "networkx": "rendering-equation dependency graph",
        "pandas": "CSV summaries for sensor bands and sampling experiments",
    },
    "visual_sequence": [
        "photon-energy-sensor-bands.png",
        "radiance-irradiance-geometry.png",
        "fresnel-refraction-beer.png",
        "brdf-energy-lobes.png",
        "brdf-lobe-explorer.html",
        "rendering-equation-visibility-geometry.png",
        "monte-carlo-hemisphere-convergence.png",
        "path-traced-diffuse-synthetic-room.png",
    ],
}
storyboard_path = remember(save_json(storyboard, TOPIC, "visual-storyboard.json"), check_paths)
display_artifact(storyboard_path)


## 1. Photons And Spectral Bookkeeping

The chapter starts with photons because they make energy accounting concrete. In graphics a photon is usually an energy packet that travels along a ray and carries a wavelength. That model is simpler than full wave optics, but it is enough to motivate spectral radiometry: instead of asking for energy at one exact wavelength, we ask for energy density over wavelength bins.

The plot below has three jobs. First, it shows the `h c / lambda` dependence, so blue photons carry more energy than red photons. Second, it shows a synthetic spectral distribution split into finite wavelength bins, which is how a measurement device would see the continuum. Third, it applies simple RGB bandpass filters like the sensor model in the chapter. The RGB bars are not a color-science model; they are a debugging model for how wavelength-dependent packets become three counters.


In [ ]:
wavelengths = np.linspace(400.0, 700.0, 601)
energies_j = photon_energy_joule(wavelengths)
electron_volt = 1.602176634e-19
energies_ev = energies_j / electron_volt

spectrum = (
    0.65 * np.exp(-0.5 * ((wavelengths - 455.0) / 22.0) ** 2)
    + 1.00 * np.exp(-0.5 * ((wavelengths - 545.0) / 38.0) ** 2)
    + 0.45 * np.exp(-0.5 * ((wavelengths - 640.0) / 28.0) ** 2)
)
spectrum /= np.trapezoid(spectrum, wavelengths)

bins = np.arange(400.0, 725.0, 25.0)
bin_rows = []
for lo, hi in zip(bins[:-1], bins[1:]):
    mask = (wavelengths >= lo) & (wavelengths <= hi)
    bin_energy_density = float(np.trapezoid(spectrum[mask], wavelengths[mask]) / (hi - lo))
    bin_rows.append({"lambda_min_nm": lo, "lambda_max_nm": hi, "mean_spectral_density": bin_energy_density})

bands = {
    "blue": (400.0, 500.0, "#3b82f6"),
    "green": (500.0, 600.0, "#22c55e"),
    "red": (600.0, 700.0, "#ef4444"),
}
band_rows = []
for name, (lo, hi, color) in bands.items():
    mask = (wavelengths >= lo) & (wavelengths < hi if hi < 700.0 else wavelengths <= hi)
    energy_fraction = float(np.trapezoid(spectrum[mask], wavelengths[mask]))
    mean_photon_energy = float(np.trapezoid(energies_ev[mask] * spectrum[mask], wavelengths[mask]) / np.trapezoid(spectrum[mask], wavelengths[mask]))
    band_rows.append({
        "band": name,
        "lambda_min_nm": lo,
        "lambda_max_nm": hi,
        "synthetic_energy_fraction": energy_fraction,
        "mean_photon_energy_eV": mean_photon_energy,
    })

fig, axes = plt.subplots(1, 3, figsize=(12.5, 3.6))
axes[0].plot(wavelengths, energies_ev, color=PALETTE["blue"], lw=2)
axes[0].set_title("photon energy q = h c / lambda")
axes[0].set_xlabel("wavelength (nm)")
axes[0].set_ylabel("energy per photon (eV)")
axes[0].grid(True, color="#d7dde5")
axes[0].annotate("blue photon\nmore energy", xy=(430, energies_ev[wavelengths.searchsorted(430)]), xytext=(455, 3.05), arrowprops={"arrowstyle": "->"})
axes[0].annotate("red photon\nless energy", xy=(660, energies_ev[wavelengths.searchsorted(660)]), xytext=(585, 2.15), arrowprops={"arrowstyle": "->"})

axes[1].plot(wavelengths, spectrum, color=PALETTE["ink"], lw=2, label="synthetic spectrum density")
centers = [(row["lambda_min_nm"] + row["lambda_max_nm"]) / 2.0 for row in bin_rows]
heights = [row["mean_spectral_density"] for row in bin_rows]
axes[1].bar(centers, heights, width=23, color="#a7c7e7", edgecolor="white", alpha=0.75, label="25 nm bin density")
axes[1].set_title("spectral energy is density over wavelength")
axes[1].set_xlabel("wavelength (nm)")
axes[1].set_ylabel("relative density")
axes[1].legend(frameon=False, fontsize=8)
axes[1].grid(True, color="#d7dde5")

for name, (lo, hi, color) in bands.items():
    axes[2].fill_between([lo, hi], [1, 1], color=color, alpha=0.2)
    axes[2].plot([lo, lo, hi, hi], [0, 1, 1, 0], color=color, lw=2, label=f"{name} passband")
axes[2].bar([0.2, 0.5, 0.8], [row["synthetic_energy_fraction"] for row in band_rows], width=0.16, color=[bands[row["band"]][2] for row in band_rows])
axes[2].set_xlim(380, 720)
axes[2].set_ylim(0, 1.05)
axes[2].set_title("bandpass filters become RGB counters")
axes[2].set_xlabel("wavelength (nm); bars use same fractions")
axes[2].set_ylabel("filter response / energy fraction")
axes[2].legend(frameon=False, fontsize=7, loc="upper left")
axes[2].grid(True, color="#d7dde5")
fig.tight_layout()

photon_fig = remember(save_matplotlib(fig, TOPIC, "photon-energy-sensor-bands.png"), figure_paths)
close(fig)

sensor_table = remember(save_table_csv(band_rows, TOPIC, "photon-sensor-band-counts.csv"), table_paths)
photon_checks = {
    "energy_400nm_eV": float(energies_ev[0]),
    "energy_700nm_eV": float(energies_ev[-1]),
    "energy_decreases_with_wavelength": bool(np.all(np.diff(energies_j) < 0.0)),
    "band_energy_fraction_sum": float(sum(row["synthetic_energy_fraction"] for row in band_rows)),
    "positive_band_fractions": bool(all(row["synthetic_energy_fraction"] > 0.0 for row in band_rows)),
}
photon_check_path = remember(save_json(photon_checks, TOPIC, "radiometry-photon-checks.json"), check_paths)

display_artifact(photon_fig, width=920)
display_artifact(sensor_table)
photon_checks


## 2. Radiometry: Why Radiance Is The Quantity Renderers Carry

Energy and power are extensive totals. Irradiance and radiance are densities. The distinction matters because a renderer almost never stores every photon; it stores estimates of densities and moves them through geometry.

Radiance is the central quantity because it stays constant along a ray in empty space. The detector receives less power per unit area as distance grows, but it also sees a larger patch of the source. Those two factors cancel. Irradiance does not carry direction by itself; it is the integral of incoming radiance over directions, with the `cos(theta)` projection term because a tilted surface presents less area to the incoming beam.


In [ ]:
theta, phi, r_symbol = sp.symbols("theta phi r", positive=True)
hemisphere_cos_integral = sp.integrate(
    sp.integrate(sp.cos(theta) * sp.sin(theta), (theta, 0, sp.pi / 2)),
    (phi, 0, 2 * sp.pi),
)
lambertian_reflectance_identity = sp.simplify((r_symbol / sp.pi) * hemisphere_cos_integral)

radii = np.linspace(0.65, 4.0, 300)
visible_patch_area = radii**2
inverse_square_arrival = 1.0 / radii**2
radiance_reading = visible_patch_area * inverse_square_arrival
angle_grid = np.linspace(0.0, np.pi / 2.0, 1000)
hemisphere_numeric = float(2.0 * np.pi * np.trapezoid(np.cos(angle_grid) * np.sin(angle_grid), angle_grid))

fig = plt.figure(figsize=(12.5, 3.8))
gs = fig.add_gridspec(1, 3, width_ratios=[1.1, 1.0, 1.0])
ax0 = fig.add_subplot(gs[0, 0])
ax0.set_title("radiance detector: distance cancellation")
ax0.set_aspect("equal")
ax0.axis("off")
source = np.array([0.0, 0.0])
patch = plt.Rectangle((-0.18, -0.22), 0.36, 0.44, facecolor="#f3d58b", edgecolor=PALETTE["ink"])
ax0.add_patch(patch)
for dist, color, label in [(1.25, PALETTE["blue"], "near detector"), (2.55, PALETTE["red"], "far detector")]:
    detector = np.array([dist, 0.0])
    ax0.plot([source[0], detector[0]], [0.18, 0.08], color=color, lw=1.4)
    ax0.plot([source[0], detector[0]], [-0.18, -0.08], color=color, lw=1.4)
    ax0.scatter([detector[0]], [detector[1]], s=70, color=color)
    ax0.text(detector[0] - 0.28, detector[1] + 0.22, label, fontsize=8, color=color)
ax0.text(-0.32, 0.33, "same surface\nradiance", fontsize=8)
ax0.text(1.1, -0.55, "area seen grows like r^2;\narrival per area falls like 1/r^2", fontsize=8)
ax0.set_xlim(-0.55, 3.05)
ax0.set_ylim(-0.8, 0.85)

ax1 = fig.add_subplot(gs[0, 1])
ax1.plot(radii, visible_patch_area / visible_patch_area[0], label="sampled source area", color=PALETTE["gold"])
ax1.plot(radii, inverse_square_arrival / inverse_square_arrival[0], label="inverse-square arrival", color=PALETTE["red"])
ax1.plot(radii, radiance_reading / radiance_reading[0], label="radiance reading", color=PALETTE["blue"], lw=2.5)
style_axis(ax1, "why radiance is invariant", xlabel="distance scale r", ylabel="relative factor")
ax1.legend(frameon=False, fontsize=8)

ax2 = fig.add_subplot(gs[0, 2])
ax2.plot(np.degrees(angle_grid), np.cos(angle_grid) * np.sin(angle_grid), color=PALETTE["teal"], lw=2)
ax2.fill_between(np.degrees(angle_grid), 0, np.cos(angle_grid) * np.sin(angle_grid), color=PALETTE["teal"], alpha=0.18)
style_axis(ax2, "irradiance integrand for constant L", xlabel="polar angle theta (degrees)", ylabel="cos(theta) sin(theta)")
ax2.text(0.05, 0.9, f"2 pi integral = {hemisphere_numeric:.6f}", transform=ax2.transAxes, fontsize=9)
fig.tight_layout()

radiance_fig = remember(save_matplotlib(fig, TOPIC, "radiance-irradiance-geometry.png"), figure_paths)
close(fig)

radiance_checks = {
    "symbolic_hemisphere_cos_integral": str(hemisphere_cos_integral),
    "symbolic_lambertian_reflectance": str(lambertian_reflectance_identity),
    "numeric_hemisphere_cos_integral": hemisphere_numeric,
    "target_pi": float(np.pi),
    "radiance_distance_product_std": float(np.std(radiance_reading)),
    "radiance_product_constant": bool(np.std(radiance_reading) < 1e-12),
}
radiance_check_path = remember(save_json(radiance_checks, TOPIC, "radiance-invariants.json"), check_paths)

display_artifact(radiance_fig, width=920)
radiance_checks


## 3. Smooth Interfaces: Fresnel, Refraction, And Beer Attenuation

For a smooth metal, a photon reflects or is absorbed. For a smooth dielectric, a photon reflects or refracts, and transmitted light can be attenuated inside the material. A path tracer often turns those fractions into random branch choices: compare a uniform random number with Fresnel reflectance, then either follow the reflected ray or the transmitted ray.

The figure below keeps the three interface rules side by side. Schlick's approximation gives the angle-dependent reflectance. Snell's law gives the refracted direction when the square-root term is nonnegative. Beer attenuation colors a ray by path length through the medium. These are local rules, but small mistakes in them show up as global rendering errors: missing total internal reflection, glass that gains energy, or tinted objects whose color does not depend on thickness.


In [ ]:
angles_deg = np.linspace(0.0, 89.9, 500)
cos_values = np.cos(np.radians(angles_deg))
material_indices = {"water n=1.33": 1.33, "glass n=1.50": 1.50, "diamond n=2.42": 2.42}
metal_r0_rgb = np.array([0.95, 0.72, 0.36])
metal_curves = schlick(cos_values[:, None], metal_r0_rgb[None, :])

incident_40 = normalize(np.array([math.sin(math.radians(40.0)), -math.cos(math.radians(40.0)), 0.0]))
transmitted_40, k_40 = refract_direction(incident_40, np.array([0.0, 1.0, 0.0]), 1.0, 1.5)
cos_phi_40 = -float(np.dot(transmitted_40, np.array([0.0, 1.0, 0.0])))
snell_residual = abs(1.0 * math.sin(math.radians(40.0)) - 1.5 * math.sqrt(max(0.0, 1.0 - cos_phi_40**2)))
critical_angle_glass_air = math.degrees(math.asin(1.0 / 1.5))
inside_50 = normalize(np.array([math.sin(math.radians(50.0)), math.cos(math.radians(50.0)), 0.0]))
tir_direction, tir_k = refract_direction(inside_50, np.array([0.0, -1.0, 0.0]), 1.5, 1.0)

distances = np.linspace(0.0, 5.0, 240)
green_glass_a = np.array([0.78, 0.96, 0.68])
beer_rgb = beer_transmission(green_glass_a, distances)

fig, axes = plt.subplots(1, 3, figsize=(13.0, 3.9))
for label, nt in material_indices.items():
    r0 = r0_from_indices(1.0, nt)
    axes[0].plot(angles_deg, schlick(cos_values, r0), lw=2, label=f"{label}, R0={r0:.3f}")
for channel, color, name in zip(range(3), ["#ef4444", "#22c55e", "#3b82f6"], ["metal red", "metal green", "metal blue"]):
    axes[0].plot(angles_deg, metal_curves[:, channel], color=color, ls="--", lw=1.5, label=name)
style_axis(axes[0], "Schlick Fresnel approaches 1 at grazing", xlabel="incident angle theta", ylabel="reflectance R(theta)")
axes[0].set_ylim(-0.02, 1.03)
axes[0].legend(frameon=False, fontsize=7, ncol=1)

ax = axes[1]
ax.set_title("Snell refraction and total internal reflection")
ax.axhline(0, color=PALETTE["ink"], lw=1.2)
ax.plot([0, 0], [-1.0, 1.0], color="#9aa5b1", lw=1, ls=":")
ax.text(0.04, 0.84, "air", fontsize=9)
ax.text(0.04, -0.92, "glass", fontsize=9)
inc2 = np.array([-incident_40[0], -incident_40[1]])
tr2 = np.array([transmitted_40[0], transmitted_40[1]])
ax.arrow(-inc2[0], -inc2[1], inc2[0], inc2[1], head_width=0.045, head_length=0.07, color=PALETTE["blue"], length_includes_head=True)
ax.arrow(0, 0, tr2[0], tr2[1], head_width=0.045, head_length=0.07, color=PALETTE["teal"], length_includes_head=True)
ax.text(-0.78, 0.74, "incident 40 deg", color=PALETTE["blue"], fontsize=8)
ax.text(0.28, -0.58, "transmitted", color=PALETTE["teal"], fontsize=8)
ax.plot([-0.65, 0.65], [-0.64, 0.64], color=PALETTE["red"], ls="--", lw=1.6)
ax.text(-0.95, -0.86, f"glass to air above {critical_angle_glass_air:.1f} deg: TIR", color=PALETTE["red"], fontsize=8)
ax.set_xlim(-1.05, 1.05)
ax.set_ylim(-1.05, 1.05)
ax.set_aspect("equal")
ax.axis("off")

for c, color, label in [(0, "#ef4444", "red"), (1, "#22c55e", "green"), (2, "#3b82f6", "blue")]:
    axes[2].plot(distances, beer_rgb[:, c], color=color, lw=2, label=label)
style_axis(axes[2], "Beer attenuation I(s) = I(0) a**s", xlabel="distance in medium", ylabel="remaining channel fraction")
axes[2].set_ylim(0.0, 1.02)
axes[2].legend(frameon=False, fontsize=8)
fig.tight_layout()

fresnel_fig = remember(save_matplotlib(fig, TOPIC, "fresnel-refraction-beer.png"), figure_paths)
close(fig)

fresnel_checks = {
    "water_R0": r0_from_indices(1.0, 1.33),
    "glass_R0": r0_from_indices(1.0, 1.5),
    "diamond_R0": r0_from_indices(1.0, 2.42),
    "schlick_bounds_hold": bool(all(np.min(schlick(cos_values, r0_from_indices(1.0, nt))) >= -1e-12 and np.max(schlick(cos_values, r0_from_indices(1.0, nt))) <= 1.0 + 1e-12 for nt in material_indices.values())),
    "snell_residual_air_to_glass_40deg": float(snell_residual),
    "transmitted_direction_norm": float(np.linalg.norm(transmitted_40)),
    "critical_angle_glass_to_air_deg": float(critical_angle_glass_air),
    "tir_square_root_term_for_50deg_glass_to_air": float(tir_k),
    "tir_detected": bool(tir_direction is None and tir_k < 0.0),
    "beer_channels_monotone": bool(np.all(np.diff(beer_rgb, axis=0) <= 1e-12)),
}
fresnel_check_path = remember(save_json(fresnel_checks, TOPIC, "fresnel-refraction-beer-checks.json"), check_paths)

display_artifact(fresnel_fig, width=940)
fresnel_checks


## 4. BRDFs: Local Scattering With A Global Energy Budget

A BRDF is a ratio: outgoing surface radiance divided by incident irradiance from a direction. It is not simply a color. Its units and normalization matter because the rendering equation integrates it over the incident hemisphere.

The easiest BRDF to normalize is Lambertian reflection. If a surface reflects fraction `r` of the incident energy, then its BRDF is `r / pi`, not `r`. The extra `pi` comes from integrating `cos(theta)` over the hemisphere. Glossy lobes can be much narrower than Lambertian reflection while conserving the same total reflected energy, provided the lobe height increases with the right normalization.

The static plot compares normalized cosine-power lobes. The HTML artifact turns one glossy lobe into a rotatable 3D surface; the radius is proportional to BRDF value, so the narrow tall shape is visible without pretending it is geometric microstructure.


In [ ]:
theta_grid = np.linspace(0.0, np.pi / 2.0, 1400)
reflectance = 0.72
exponents = [0, 4, 16, 64]
lobe_rows = []
for exponent in exponents:
    rho = normalized_cosine_lobe(theta_grid, reflectance, exponent)
    energy = directional_hemispherical_reflectance(theta_grid, rho)
    lobe_rows.append({"exponent": exponent, "peak_brdf": float(rho.max()), "integrated_reflectance": energy, "energy_error": float(energy - reflectance)})

fig = plt.figure(figsize=(12.5, 3.9))
ax0 = fig.add_subplot(1, 3, 1, projection="polar")
for row in lobe_rows:
    exponent = row["exponent"]
    signed_theta = np.linspace(-np.pi / 2.0, np.pi / 2.0, 700)
    rho_signed = normalized_cosine_lobe(np.abs(signed_theta), reflectance, exponent)
    ax0.plot(signed_theta, rho_signed / max(rho_signed), lw=2, label=f"m={exponent}")
ax0.set_theta_zero_location("N")
ax0.set_theta_direction(-1)
ax0.set_thetamin(-90)
ax0.set_thetamax(90)
ax0.set_title("normalized lobe shape")
ax0.set_rticks([0.5, 1.0])
ax0.legend(loc="lower center", bbox_to_anchor=(0.5, -0.22), ncol=4, frameon=False, fontsize=8)

ax1 = fig.add_subplot(1, 3, 2)
ax1.bar([str(row["exponent"]) for row in lobe_rows], [row["integrated_reflectance"] for row in lobe_rows], color=PALETTE["teal"], alpha=0.78)
ax1.axhline(reflectance, color=PALETTE["red"], lw=2, label="target reflectance")
style_axis(ax1, "directional hemispherical reflectance", xlabel="cosine-power exponent", ylabel="integrated R")
ax1.set_ylim(0, 1.0)
ax1.legend(frameon=False, fontsize=8)

ax2 = fig.add_subplot(1, 3, 3)
for exponent in exponents:
    rho = normalized_cosine_lobe(theta_grid, reflectance, exponent)
    contribution = rho * np.cos(theta_grid) * np.sin(theta_grid)
    ax2.plot(np.degrees(theta_grid), contribution / np.max(contribution), lw=2, label=f"m={exponent}")
style_axis(ax2, "where the reflected energy lives", xlabel="outgoing polar angle theta_o", ylabel="normalized integrand")
ax2.legend(frameon=False, fontsize=8)
fig.tight_layout()

brdf_fig = remember(save_matplotlib(fig, TOPIC, "brdf-energy-lobes.png"), figure_paths)
close(fig)

theta_s = np.linspace(0.0, np.pi / 2.0, 58)
phi_s = np.linspace(0.0, 2.0 * np.pi, 96)
TH, PH = np.meshgrid(theta_s, phi_s, indexing="ij")
rho_surface = normalized_cosine_lobe(TH, reflectance, 16)
radius = rho_surface / rho_surface.max()
X = radius * np.sin(TH) * np.cos(PH)
Y = radius * np.sin(TH) * np.sin(PH)
Z = radius * np.cos(TH)
fig3d = go.Figure(data=[go.Surface(x=X, y=Y, z=Z, surfacecolor=rho_surface, colorscale="Viridis", showscale=True, colorbar={"title": "BRDF"})])
fig3d.update_layout(
    title="BRDF lobe explorer: normalized cosine-power glossy lobe",
    scene={"xaxis_title": "x", "yaxis_title": "y", "zaxis_title": "surface normal", "aspectmode": "data"},
    margin={"l": 0, "r": 0, "b": 0, "t": 45},
)
brdf_html = remember(save_plotly_html(fig3d, TOPIC, "brdf-lobe-explorer.html"), html_paths)

lobe_table = remember(save_table_csv(lobe_rows, TOPIC, "brdf-lobe-energy.csv"), table_paths)
brdf_checks = {
    "target_reflectance": reflectance,
    "max_abs_energy_error": float(max(abs(row["energy_error"]) for row in lobe_rows)),
    "lambertian_brdf_value": float(reflectance / np.pi),
    "all_lobes_conserve_energy_within_tolerance": bool(max(abs(row["energy_error"]) for row in lobe_rows) < 1e-4),
    "no_lobe_reflectance_exceeds_one": bool(all(row["integrated_reflectance"] <= 1.0 + 1e-4 for row in lobe_rows)),
}
brdf_check_path = remember(save_json(brdf_checks, TOPIC, "brdf-energy-checks.json"), check_paths)

display_artifact(brdf_fig, width=920)
display_artifact(brdf_html, width="100%", height=520)
display_artifact(lobe_table)
brdf_checks


## 5. The Rendering Equation As A Visibility-Weighted Transport Rule

The directional form of the rendering equation says: collect incident radiance from every direction, multiply by the BRDF and the projected-area factor, and integrate. The area form makes a different dependency visible: light arriving at point `x` came from another visible surface point `x'`, and the solid angle of that small patch contributes a `cos(theta') / distance^2` factor.

That area-form integrand is where many bugs hide. A blocked segment should contribute zero. A back-facing patch should contribute zero. A far patch should contribute less because it subtends less solid angle. The graph in the figure is a proof scaffold: each incoming dependency must be present for surface radiance to be updated correctly.


In [ ]:
def orient(a: np.ndarray, b: np.ndarray, c: np.ndarray) -> float:
    return float((b[0] - a[0]) * (c[1] - a[1]) - (b[1] - a[1]) * (c[0] - a[0]))


def segments_intersect(a: np.ndarray, b: np.ndarray, c: np.ndarray, d: np.ndarray) -> bool:
    o1 = orient(a, b, c)
    o2 = orient(a, b, d)
    o3 = orient(c, d, a)
    o4 = orient(c, d, b)
    return (o1 * o2 < 0.0) and (o3 * o4 < 0.0)


def geometry_factor(x: np.ndarray, nx_normal: np.ndarray, xp: np.ndarray, nxp_normal: np.ndarray, occluder: tuple[np.ndarray, np.ndarray] | None = None) -> tuple[float, dict[str, float | bool]]:
    delta = xp - x
    distance2 = float(np.dot(delta, delta))
    wi = normalize(delta)
    wip = normalize(-delta)
    cos_i = max(0.0, float(np.dot(normalize(nx_normal), wi)))
    cos_p = max(0.0, float(np.dot(normalize(nxp_normal), wip)))
    visible = True if occluder is None else not segments_intersect(x, xp, occluder[0], occluder[1])
    value = (1.0 if visible else 0.0) * cos_i * cos_p / distance2
    return value, {"cos_theta_i": cos_i, "cos_theta_prime": cos_p, "distance2": distance2, "visible": visible}

x = np.array([0.0, 0.0])
xp = np.array([1.55, 1.05])
nx_normal = normalize(np.array([0.15, 1.0]))
nxp_normal = normalize(np.array([-0.35, -1.0]))
occluder = (np.array([0.72, -0.12]), np.array([0.84, 0.82]))
visible_value, visible_parts = geometry_factor(x, nx_normal, xp, nxp_normal)
occluded_value, occluded_parts = geometry_factor(x, nx_normal, xp, nxp_normal, occluder)
backface_value, backface_parts = geometry_factor(x, -nx_normal, xp, nxp_normal)

fig = plt.figure(figsize=(12.8, 4.2))
gs = fig.add_gridspec(1, 2, width_ratios=[1.05, 1.25])
ax0 = fig.add_subplot(gs[0, 0])
ax0.set_title("area-form transport geometry")
ax0.set_aspect("equal")
ax0.grid(True, color="#d7dde5")
ax0.scatter([x[0], xp[0]], [x[1], xp[1]], s=90, color=[PALETTE["blue"], PALETTE["gold"]], zorder=3)
ax0.text(x[0] - 0.18, x[1] - 0.18, "x", fontsize=11, color=PALETTE["blue"])
ax0.text(xp[0] + 0.05, xp[1] + 0.04, "x'", fontsize=11, color=PALETTE["gold"])
ax0.arrow(x[0], x[1], 0.33 * nx_normal[0], 0.33 * nx_normal[1], color=PALETTE["blue"], width=0.008, head_width=0.055, length_includes_head=True)
ax0.arrow(xp[0], xp[1], 0.33 * nxp_normal[0], 0.33 * nxp_normal[1], color=PALETTE["gold"], width=0.008, head_width=0.055, length_includes_head=True)
ax0.plot([x[0], xp[0]], [x[1], xp[1]], color=PALETTE["teal"], lw=2, label="visible segment")
ax0.plot([occluder[0][0], occluder[1][0]], [occluder[0][1], occluder[1][1]], color=PALETTE["red"], lw=5, solid_capstyle="round", label="occluder")
ax0.text(0.35, 0.55, "visibility v(x,x')", color=PALETTE["red"], fontsize=8)
ax0.set_xlim(-0.45, 2.05)
ax0.set_ylim(-0.38, 1.45)
ax0.legend(frameon=False, fontsize=8, loc="lower right")

ax1 = fig.add_subplot(gs[0, 1])
dep = nx.DiGraph()
edges = [
    ("incoming radiance\nL_s(x', x-x')", "integrand"),
    ("BRDF\nrho(k_i,k_o)", "integrand"),
    ("cos theta_i", "integrand"),
    ("cos theta'", "integrand"),
    ("visibility\nv(x,x')", "integrand"),
    ("distance^-2", "integrand"),
    ("integrand", "surface radiance\nL_s(x,k_o)"),
    ("emission", "surface radiance\nL_s(x,k_o)"),
]
dep.add_edges_from(edges)
pos = {
    "incoming radiance\nL_s(x', x-x')": (0.0, 0.8),
    "BRDF\nrho(k_i,k_o)": (0.0, 0.5),
    "cos theta_i": (0.0, 0.2),
    "cos theta'": (0.0, -0.1),
    "visibility\nv(x,x')": (0.0, -0.4),
    "distance^-2": (0.0, -0.7),
    "integrand": (1.05, 0.05),
    "surface radiance\nL_s(x,k_o)": (2.15, 0.22),
    "emission": (1.05, 0.82),
}
nx.draw_networkx_nodes(dep, pos, ax=ax1, node_color="#e8f0fe", edgecolors=PALETTE["ink"], node_size=2200)
nx.draw_networkx_edges(dep, pos, ax=ax1, arrows=True, arrowstyle="-|>", arrowsize=13, edge_color="#607080")
nx.draw_networkx_labels(dep, pos, ax=ax1, font_size=8)
ax1.set_title("rendering equation dependency scaffold")
ax1.axis("off")
fig.tight_layout()

transport_fig = remember(save_matplotlib(fig, TOPIC, "rendering-equation-visibility-geometry.png"), figure_paths)
close(fig)

transport_checks = {
    "visible_geometry_factor": visible_value,
    "occluded_geometry_factor": occluded_value,
    "backface_geometry_factor": backface_value,
    "visible_parts": visible_parts,
    "occluded_parts": occluded_parts,
    "positive_when_visible": bool(visible_value > 0.0),
    "zero_when_occluded": bool(occluded_value == 0.0),
    "zero_when_backfacing": bool(backface_value == 0.0),
}
transport_check_path = remember(save_json(transport_checks, TOPIC, "rendering-equation-geometry-checks.json"), check_paths)

display_artifact(transport_fig, width=940)
transport_checks


## 6. Monte Carlo Rendering Equation Estimates

Once lighting is an integral, random sampling becomes a rendering algorithm. The estimator has the same shape every time: evaluate the integrand at random directions and divide by the probability density that produced those directions. The PDF can be any valid PDF over the domain, but it controls variance.

The experiment estimates irradiance from a nonuniform synthetic incident radiance field,

`L(k) = 1 + 0.6 k_x + 0.3 k_z`,

on the upper hemisphere. The exact integral is `1.2 pi`: the sideways term integrates to zero and the `k_z` term contributes `0.3 * 2 pi / 3`. Uniform hemisphere sampling is valid but wastes many samples near grazing angles where the cosine factor is small. Cosine-weighted sampling follows the projected-area term and has lower error for the same sample count.


In [ ]:
def sample_uniform_hemisphere(rng: np.random.Generator, count: int) -> np.ndarray:
    z = rng.random(count)
    phi = 2.0 * np.pi * rng.random(count)
    r = np.sqrt(np.maximum(0.0, 1.0 - z * z))
    return np.column_stack([r * np.cos(phi), r * np.sin(phi), z])


def sample_cosine_hemisphere(rng: np.random.Generator, count: int) -> np.ndarray:
    u1 = rng.random(count)
    u2 = rng.random(count)
    r = np.sqrt(u1)
    phi = 2.0 * np.pi * u2
    z = np.sqrt(np.maximum(0.0, 1.0 - u1))
    return np.column_stack([r * np.cos(phi), r * np.sin(phi), z])


def incident_radiance(directions: np.ndarray) -> np.ndarray:
    return 1.0 + 0.6 * directions[:, 0] + 0.3 * directions[:, 2]


def estimate_irradiance(rng: np.random.Generator, count: int, method: str) -> float:
    if method == "uniform hemisphere":
        dirs = sample_uniform_hemisphere(rng, count)
        pdf = np.full(count, 1.0 / (2.0 * np.pi))
    elif method == "cosine weighted":
        dirs = sample_cosine_hemisphere(rng, count)
        pdf = dirs[:, 2] / np.pi
    else:
        raise ValueError(method)
    values = incident_radiance(dirs) * dirs[:, 2] / pdf
    return float(np.mean(values))

true_irradiance = float(1.2 * np.pi)
sample_counts = [1, 2, 4, 8, 16, 32, 64, 128, 256, 512, 1024, 2048]
trial_count = 96
rows = []
for method in ["uniform hemisphere", "cosine weighted"]:
    for count in sample_counts:
        estimates = []
        for trial in range(trial_count):
            rng = np.random.default_rng(10_000 + 173 * trial + count + (0 if method.startswith("uniform") else 50_000))
            estimates.append(estimate_irradiance(rng, count, method))
        estimates = np.asarray(estimates)
        rows.append({
            "method": method,
            "samples": count,
            "mean_estimate": float(estimates.mean()),
            "bias": float(estimates.mean() - true_irradiance),
            "rmse": float(np.sqrt(np.mean((estimates - true_irradiance) ** 2))),
            "stddev": float(estimates.std(ddof=1)),
            "true_irradiance": true_irradiance,
        })
mc_df = pd.DataFrame(rows)
mc_table = remember(save_table_csv(mc_df.to_dict("records"), TOPIC, "monte-carlo-estimates.csv"), table_paths)

fig, axes = plt.subplots(1, 2, figsize=(12.2, 3.9))
for method, color in [("uniform hemisphere", PALETTE["red"]), ("cosine weighted", PALETTE["blue"])]:
    sub = mc_df[mc_df["method"] == method]
    axes[0].plot(sub["samples"], sub["mean_estimate"], marker="o", color=color, label=method)
    axes[1].plot(sub["samples"], sub["rmse"], marker="o", color=color, label=method)
axes[0].axhline(true_irradiance, color=PALETTE["ink"], lw=1.5, ls="--", label="exact 1.2 pi")
axes[0].set_xscale("log", base=2)
axes[1].set_xscale("log", base=2)
axes[1].set_yscale("log")
style_axis(axes[0], "estimator mean", xlabel="samples per estimate", ylabel="irradiance estimate")
style_axis(axes[1], "root mean square error", xlabel="samples per estimate", ylabel="RMSE")
axes[0].legend(frameon=False, fontsize=8)
axes[1].legend(frameon=False, fontsize=8)
fig.tight_layout()

mc_fig = remember(save_matplotlib(fig, TOPIC, "monte-carlo-hemisphere-convergence.png"), figure_paths)
close(fig)

final_uniform = mc_df[(mc_df["method"] == "uniform hemisphere") & (mc_df["samples"] == sample_counts[-1])].iloc[0]
final_cosine = mc_df[(mc_df["method"] == "cosine weighted") & (mc_df["samples"] == sample_counts[-1])].iloc[0]
mc_checks = {
    "true_irradiance": true_irradiance,
    "final_uniform_mean": float(final_uniform["mean_estimate"]),
    "final_cosine_mean": float(final_cosine["mean_estimate"]),
    "final_uniform_rmse": float(final_uniform["rmse"]),
    "final_cosine_rmse": float(final_cosine["rmse"]),
    "cosine_rmse_less_than_uniform": bool(final_cosine["rmse"] < final_uniform["rmse"]),
    "final_estimates_close": bool(abs(final_uniform["mean_estimate"] - true_irradiance) < 0.08 and abs(final_cosine["mean_estimate"] - true_irradiance) < 0.08),
}
mc_check_path = remember(save_json(mc_checks, TOPIC, "monte-carlo-sampling-checks.json"), check_paths)

display_artifact(mc_fig, width=900)
display_artifact(mc_table)
mc_checks


## 7. Applied lab: A Tiny Synthetic Path Tracer

The chapter's brute-force photon tracer is easiest to understand forward in time, but almost all practical ray/path tracing starts at the sensor and walks paths backward into the scene. The following lab implements a deliberately small version: a pinhole camera, two diffuse spheres, a diffuse floor and wall, an environment light, cosine-weighted BRDF sampling, and a finite bounce limit.

This is not a production renderer and it avoids caustics, texture maps, external images, and measured materials. Its purpose is narrower: expose the recursion in the rendering equation and make sampling noise inspectable. The left image uses one sample per pixel; the right image uses more samples with the same transport rules.


In [ ]:
def intersect_sphere(origin: np.ndarray, direction: np.ndarray, center: np.ndarray, radius: float) -> tuple[float, np.ndarray] | None:
    oc = origin - center
    b = float(np.dot(oc, direction))
    c = float(np.dot(oc, oc) - radius * radius)
    disc = b * b - c
    if disc < 0.0:
        return None
    root = math.sqrt(disc)
    t = -b - root
    if t < 1e-4:
        t = -b + root
    if t < 1e-4:
        return None
    point = origin + t * direction
    return t, normalize(point - center)


def intersect_plane(origin: np.ndarray, direction: np.ndarray, point: np.ndarray, normal: np.ndarray) -> tuple[float, np.ndarray] | None:
    n = normalize(normal)
    denom = float(np.dot(direction, n))
    if abs(denom) < 1e-6:
        return None
    t = float(np.dot(point - origin, n) / denom)
    if t < 1e-4:
        return None
    return t, n if denom < 0.0 else -n

scene_objects = [
    {"kind": "sphere", "center": np.array([-0.65, 0.48, -0.55]), "radius": 0.48, "albedo": np.array([0.86, 0.28, 0.18]), "emission": np.zeros(3)},
    {"kind": "sphere", "center": np.array([0.62, 0.34, -0.95]), "radius": 0.34, "albedo": np.array([0.22, 0.55, 0.88]), "emission": np.zeros(3)},
    {"kind": "plane", "point": np.array([0.0, 0.0, 0.0]), "normal": np.array([0.0, 1.0, 0.0]), "albedo": np.array([0.74, 0.72, 0.66]), "emission": np.zeros(3)},
    {"kind": "plane", "point": np.array([0.0, 0.0, -2.2]), "normal": np.array([0.0, 0.0, 1.0]), "albedo": np.array([0.70, 0.76, 0.72]), "emission": np.zeros(3)},
]


def intersect_scene(origin: np.ndarray, direction: np.ndarray):
    best = None
    for obj in scene_objects:
        if obj["kind"] == "sphere":
            hit = intersect_sphere(origin, direction, obj["center"], obj["radius"])
        else:
            hit = intersect_plane(origin, direction, obj["point"], obj["normal"])
        if hit is None:
            continue
        t, normal = hit
        if best is None or t < best["t"]:
            best = {"t": t, "point": origin + t * direction, "normal": normal, "albedo": obj["albedo"], "emission": obj["emission"]}
    return best


def environment_radiance_path(direction: np.ndarray) -> np.ndarray:
    z = max(0.0, direction[1])
    horizon = np.array([0.92, 0.94, 0.98])
    sky = np.array([0.45, 0.62, 0.95])
    warm_key = np.array([1.0, 0.82, 0.55]) * max(0.0, float(np.dot(direction, normalize(np.array([-0.35, 0.55, 0.75]))))) ** 12
    return (1.0 - z) * horizon + z * sky + 1.7 * warm_key


def trace_path(origin: np.ndarray, direction: np.ndarray, rng: np.random.Generator, max_depth: int = 4) -> np.ndarray:
    radiance = np.zeros(3)
    throughput = np.ones(3)
    o = origin.copy()
    d = normalize(direction)
    for bounce in range(max_depth):
        hit = intersect_scene(o, d)
        if hit is None:
            radiance += throughput * environment_radiance_path(d)
            break
        radiance += throughput * hit["emission"]
        throughput *= hit["albedo"]
        if bounce >= 2:
            survive = min(0.95, max(0.15, float(np.max(throughput))))
            if rng.random() > survive:
                break
            throughput /= survive
        d = cosine_sample_hemisphere(rng, hit["normal"])
        o = hit["point"] + 1e-4 * hit["normal"]
    return radiance

camera_origin = np.array([0.0, 0.74, 2.65])
camera_target = np.array([0.0, 0.42, -0.72])
camera_forward = normalize(camera_target - camera_origin)
camera_right = normalize(np.cross(camera_forward, np.array([0.0, 1.0, 0.0])))
camera_up = normalize(np.cross(camera_right, camera_forward))


def camera_direction(px: float, py: float, width: int, height: int, fov_degrees: float = 43.0) -> np.ndarray:
    aspect = width / height
    scale = math.tan(math.radians(fov_degrees) / 2.0)
    sx = (2.0 * px / width - 1.0) * aspect * scale
    sy = (1.0 - 2.0 * py / height) * scale
    return normalize(camera_forward + sx * camera_right + sy * camera_up)


def render_path_traced(width: int, height: int, samples_per_pixel: int, seed: int) -> np.ndarray:
    rng = np.random.default_rng(seed)
    image = np.zeros((height, width, 3), dtype=float)
    for j in range(height):
        for i in range(width):
            color = np.zeros(3)
            for _ in range(samples_per_pixel):
                jitter = rng.random(2)
                direction = camera_direction(i + jitter[0], j + jitter[1], width, height)
                color += trace_path(camera_origin, direction, rng)
            image[j, i] = color / samples_per_pixel
    return image

width, height = 96, 64
low_spp = 1
high_spp = 24
low_image = render_path_traced(width, height, low_spp, seed=1401)
high_image = render_path_traced(width, height, high_spp, seed=1402)


def tonemap(image: np.ndarray) -> np.ndarray:
    mapped = image / (1.0 + image)
    return np.clip(mapped, 0.0, 1.0) ** (1.0 / 2.2)

fig, axes = plt.subplots(1, 2, figsize=(9.8, 3.7))
axes[0].imshow(tonemap(low_image))
axes[0].set_title(f"{low_spp} sample per pixel")
axes[1].imshow(tonemap(high_image))
axes[1].set_title(f"{high_spp} samples per pixel")
for ax in axes:
    ax.axis("off")
fig.suptitle("synthetic diffuse path tracing: same estimator, different sample count")
fig.tight_layout()

path_trace_fig = remember(save_matplotlib(fig, TOPIC, "path-traced-diffuse-synthetic-room.png"), figure_paths)
close(fig)

luminance = high_image @ np.array([0.2126, 0.7152, 0.0722])
path_summary = {
    "scene": "synthetic diffuse floor, wall, and two spheres with environment light",
    "width": width,
    "height": height,
    "low_samples_per_pixel": low_spp,
    "high_samples_per_pixel": high_spp,
    "max_depth": 4,
    "estimated_camera_paths_high": int(width * height * high_spp),
    "high_image_finite": bool(np.isfinite(high_image).all()),
    "mean_luminance_high": float(luminance.mean()),
    "std_luminance_high": float(luminance.std()),
    "max_channel_high": float(high_image.max()),
}
path_trace_check_path = remember(save_json(path_summary, TOPIC, "path-tracing-summary.json"), check_paths)

display_artifact(path_trace_fig, width=760)
path_summary


## Applied lab prompt

Use the artifacts above as a debugging loop for a small renderer. A practical material or sampling change should answer three questions before it is trusted:

1. Does the local rule conserve or intentionally absorb energy?
2. Does the sampling estimator divide by the PDF that actually generated the direction?
3. Does the rendered image change in the expected direction without breaking the radiometric checks?

The cell below compares a few compact material/sampling choices without re-rendering the whole scene. It is a fast lab for the same questions: high reflectance raises the BRDF lobe but should not push integrated reflectance over one; cosine sampling should keep lower variance than uniform sampling for a cosine-weighted integral; Beer attenuation should darken longer paths channel by channel.


In [ ]:
lab_rows = []
for refl, exponent in [(0.35, 0), (0.72, 16), (0.95, 64)]:
    rho = normalized_cosine_lobe(theta_grid, refl, exponent)
    lab_rows.append({
        "case": f"BRDF reflectance={refl}, exponent={exponent}",
        "measured_quantity": "directional hemispherical reflectance",
        "value": directional_hemispherical_reflectance(theta_grid, rho),
        "expected": refl,
        "passes": abs(directional_hemispherical_reflectance(theta_grid, rho) - refl) < 1e-4 and refl <= 1.0,
    })

for distance in [0.5, 2.0, 4.0]:
    trans = beer_transmission(green_glass_a, np.array([distance]))[0]
    lab_rows.append({
        "case": f"Beer green glass distance={distance}",
        "measured_quantity": "mean RGB transmittance",
        "value": float(trans.mean()),
        "expected": "decreases as distance increases",
        "passes": bool(np.all(trans <= 1.0) and np.all(trans > 0.0)),
    })

lab_rows.append({
    "case": "Monte Carlo final comparison",
    "measured_quantity": "cosine RMSE / uniform RMSE",
    "value": float(final_cosine["rmse"] / final_uniform["rmse"]),
    "expected": "below 1",
    "passes": bool(final_cosine["rmse"] < final_uniform["rmse"]),
})

lab_table = remember(save_table_csv(lab_rows, TOPIC, "applied-lab-sampling-summary.csv"), table_paths)
lab_checks = {
    "all_lab_rows_pass": bool(all(row["passes"] for row in lab_rows)),
    "row_count": len(lab_rows),
}
lab_check_path = remember(save_json(lab_checks, TOPIC, "applied-lab-checks.json"), check_paths)

display_artifact(lab_table)
lab_checks


## Sanity checks

The final checks are intentionally tied to the chapter's failure modes. They do not prove a renderer is production-ready. They do catch the mistakes that most directly contradict the physics-based model introduced here: nonmonotone photon energy, missing `pi` factors, Fresnel values outside bounds, a refracted ray that violates Snell's law, BRDF lobes that do not integrate to their reflectance, visibility factors that ignore blockers, Monte Carlo estimates that forget their PDF, or generated artifacts that are blank or missing.


In [ ]:
assert photon_checks["energy_decreases_with_wavelength"]
assert abs(photon_checks["band_energy_fraction_sum"] - 1.0) < 0.03
assert radiance_checks["symbolic_hemisphere_cos_integral"] == "pi"
assert abs(radiance_checks["numeric_hemisphere_cos_integral"] - np.pi) < 1e-5
assert fresnel_checks["schlick_bounds_hold"]
assert fresnel_checks["snell_residual_air_to_glass_40deg"] < 1e-12
assert fresnel_checks["tir_detected"]
assert brdf_checks["all_lobes_conserve_energy_within_tolerance"]
assert brdf_checks["no_lobe_reflectance_exceeds_one"]
assert transport_checks["positive_when_visible"] and transport_checks["zero_when_occluded"] and transport_checks["zero_when_backfacing"]
assert mc_checks["cosine_rmse_less_than_uniform"] and mc_checks["final_estimates_close"]
assert path_summary["high_image_finite"] and path_summary["mean_luminance_high"] > 0.02 and path_summary["std_luminance_high"] > 0.01
assert lab_checks["all_lab_rows_pass"]

artifact_records = assert_artifacts([*figure_paths, *html_paths, *check_paths, *table_paths])
image_records = [assert_nonblank_image(path) for path in figure_paths]

final_sanity = {
    "chapter": CHAPTER,
    "title": TITLE,
    "source_span": {"printed_pages": PRINTED_PAGES, "pdf_pages": PDF_PAGES},
    "figure_count": len(figure_paths),
    "html_count": len(html_paths),
    "check_count_before_final": len(check_paths),
    "table_count": len(table_paths),
    "nonblank_image_count": len(image_records),
    "artifact_count_before_final": len(artifact_records),
    "core_checks": {
        "photon_energy_monotone": photon_checks["energy_decreases_with_wavelength"],
        "hemisphere_integral_pi": radiance_checks["symbolic_hemisphere_cos_integral"],
        "snell_residual": fresnel_checks["snell_residual_air_to_glass_40deg"],
        "brdf_max_energy_error": brdf_checks["max_abs_energy_error"],
        "cosine_sampling_rmse_ratio": mc_checks["final_cosine_rmse"] / mc_checks["final_uniform_rmse"],
        "path_tracer_mean_luminance": path_summary["mean_luminance_high"],
    },
    "libraries_used": storyboard["library_routing"],
    "artifacts": [book_relative(path) for path in [*figure_paths, *html_paths, *check_paths, *table_paths]],
}
final_path = save_json(final_sanity, TOPIC, "final-sanity.json")
final_record = assert_artifacts([final_path])[0]
final_sanity["final_sanity_artifact"] = final_record

display_artifact(final_path)
final_sanity


## Takeaways

- Photon thinking is useful for intuition, but renderers usually carry radiometric densities rather than individual packets.
- Radiance is the transport quantity because it stays invariant along empty-space rays; irradiance appears when radiance is integrated over projected incident directions.
- Fresnel, Snell refraction, total internal reflection, and Beer attenuation are local interface or medium rules that must conserve or absorb energy deliberately.
- A BRDF is normalized by an integral over the hemisphere. For Lambertian reflection the `1/pi` factor is not optional.
- The rendering equation is a visibility-weighted integral. Directional and area forms expose different implementation hazards.
- Monte Carlo path tracing is simple because it recursively estimates the same integral, but the PDF choice controls noise and must match the sampled directions.
